In [ ]:
import torch
import gc

# Kiểm tra xem Kaggle đã bật GPU (CUDA) thành công chưa
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)

In [ ]:
# Tải dataset kaggle_dataset.jsonl
import json
from datasets import load_dataset

dataset = load_dataset("json", data_files="kaggle_dataset.jsonl", split="train")

def map_role(example):
    for msg in example['messages']:
        if msg['role'] == 'model':
            msg['role'] = 'assistant'
    return example

dataset = dataset.map(map_role)
print(dataset[0])

In [ ]:
!pip install -U git+https://github.com/huggingface/transformers.git --no-deps
!pip install -q -U peft trl bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# TỐI ƯU 1: CHUYỂN SANG MODEL NHỎ HƠN (2 Tỷ tham số)
# Model 2B cực kỳ nhẹ, chạy mượt trên 16GB RAM mà vẫn thông minh, dư sức làm task chuyển JSON sang văn bản.
MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

In [ ]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import gc

# Dọn rác bộ nhớ trước khi train để giải phóng tối đa RAM
gc.collect()
torch.cuda.empty_cache()

# Format data to ChatML/Gemma prompt structure
def formatting_func(example):
    # TỐI ƯU 2: Cắt ngắn chuỗi văn bản nếu quá dài. 
    # Tránh việc SFTTrainer tính attention lên chuỗi 8192 tokens gây tràn VRAM.
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    # Giới hạn khoảng 2000 ký tự (tương đương ~500 tokens)
    if len(text) > 2000:
        text = text[:2000]
    return text

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="gemma-2b-it-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=20,
    max_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=args,
    formatting_func=formatting_func,
)

trainer.train()

In [ ]:
# Save LoRA adapter
trainer.model.save_pretrained("gemma-2b-it-weather-lora")
tokenizer.save_pretrained("gemma-2b-it-weather-lora")
print("Model saved successfully!")